# CoSense B1 — notebook độc lập, chỉ cần Dataset dữ liệu

**Bản này thay notebook cũ cần cosense-compartment-src. Không cần project ZIP, source Dataset hoặc clone GitHub.** Code dùng để preprocess/train/evaluate đã nằm trực tiếp trong notebook.

1. Import notebook này vào Kaggle; **Add Input Dataset được tạo từ cosense-kaggle-data.zip mới** (có cả TSV và split_manifest_nn/pv.json).
2. Bật **GPU + Internet** để tải pretrained BERT lần đầu.
3. Chạy từ trên xuống. Mặc định **NN / B1 / FULL / seed42**. Nếu muốn PV, đổi TASK='pv' trong cell cấu hình rồi chạy từ đó xuống.
4. Dữ liệu đã chia trên local: notebook kiểm tra hash và đọc đúng IDs; **không chia tập lại**.
5. Đợi `OUTPUTS_READY`: notebook tự xuất ZIP results/model và CSV metrics ngay ở `/kaggle/working`. Thiếu file thì báo lỗi.
6. Chạy cell XEM METRICS và KIỂM TRA OUTPUT cuối notebook. Save Version bằng Save & Run All để chạy version đầy đủ, hoặc Quick Save với lưu files bật nếu UI hỗ trợ; kiểm tra Output của đúng saved version.

B1 chỉ đọc expression, pool hai constituents.
NN có hai outputs ModAvg/HeadAvg; PV một output Avg. Cùng backbone/head capacity/config/manifests cho B1/B2. Giữ data version và training seed giống nhau khi so sánh.
Chưa có Kaggle runtime proof cho bản standalone mới; tests local dùng synthetic fixtures/random tiny BERT, không phải kết quả nghiên cứu.


## 1. Kiểm tra môi trường
Giữ PyTorch CUDA sẵn có. Chỉ cài dependencies thiếu hoặc ngoài compatibility range. Nếu vừa thay package đã import, Restart Session rồi chạy lại. GPU khác TPU; notebook dùng CUDA GPU.


In [ ]:
from pathlib import Path
import os, sys, json, importlib.metadata, subprocess
from packaging.version import Version
required = {'transformers': ('4.46','6','transformers==5.14.1'),
            'tokenizers': ('0.20','1','tokenizers==0.22.2'),
            'numpy': ('1.26','3','numpy>=1.26,<3'), 'pandas': ('2.2','4','pandas>=2.2,<4'),
            'scipy': ('1.13','2','scipy>=1.13,<2')}
missing = []
for name, (lo, hi, spec) in required.items():
    try:
        version = Version(importlib.metadata.version(name))
        if not Version(lo) <= version < Version(hi): missing.append(spec)
    except importlib.metadata.PackageNotFoundError: missing.append(spec)
if missing: subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
if Version(torch.__version__.split('+')[0]) < Version('2.4'):
    raise RuntimeError('Cần Kaggle image có torch>=2.4. Không tự cài torch CPU wheel.')
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Chưa bật GPU')
for package in ['torch', *required]: print(package, importlib.metadata.version(package))


## 2. Cấu hình
DATA_ROOT mặc định tự tìm trong Input; nếu nhiều bản dữ liệu matching thì đặt đường dẫn Dataset cụ thể.
TASK='nn' để chạy NN trước; TASK='pv' dùng cùng notebook với PV train riêng. Không đổi MODEL: B1/B2 có hai notebook riêng.
SMOKE=False là full run, True chỉ chạy 16 rows/1 epoch. Đổi settings thì chạy lại từ cell này xuống; không chỉ sửa chữ.


In [ ]:
NOTEBOOK_MODEL = 'B1'
MODEL = 'B1'
TASK = 'nn'  # 'pv' khi chạy particle verbs.
SEED = 42
SPLIT_SEED = 42
SMOKE = False
DATA_ROOT = Path(os.environ.get('DATA_ROOT', '/kaggle/input'))
OUTPUT_ROOT = Path(os.environ.get('OUTPUT_ROOT', '/kaggle/working/cosense_outputs'))
KAGGLE_SESSION = Path('/kaggle/input').is_dir()
PUBLISH_ROOT = Path('/kaggle/working') if KAGGLE_SESSION else OUTPUT_ROOT
if KAGGLE_SESSION and not OUTPUT_ROOT.resolve().is_relative_to(PUBLISH_ROOT.resolve()):
    raise ValueError('OUTPUT_ROOT phải nằm dưới /kaggle/working để Kaggle lưu file outputs.')
RESUME = None  # Path tới resume.pt đã Save Version; giữ sibling best.pt/tokenizer/encoder_config.
EVALUATE_LOCAL_TEST = False  # Chỉ bật sau khi đã khóa recipe trên dev.
CONFIG = dict(backbone='bert-base-uncased', revision='86b5e0934494bd15c9632b12f734a8a67f723594',
              seed=SEED, lr=2e-5, batch_size=8, accumulation=2, epochs=8, patience=2,
              max_length=128, layer=-1, smoke=SMOKE, allow_cpu=False, fallback='expression', log_every_batches=20)
if SMOKE: CONFIG.update(epochs=1, smoke_rows=16)
if TASK not in ['nn','pv'] or MODEL != NOTEBOOK_MODEL: raise ValueError('Sai TASK hoặc model của notebook.')
RUN_ROOT = OUTPUT_ROOT / ('smoke_runs' if SMOKE else 'runs') / TASK / f'B1_seed{SEED}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('CONFIRMED:', TASK, MODEL, 'SMOKE' if SMOKE else 'FULL', 'seed', SEED)
print('Data:', DATA_ROOT, '| Run:', RUN_ROOT)


## 3. Code dùng chung — đã nằm trong notebook
Các cells dưới chứa code đọc/validate dữ liệu, alignment, metrics, tokenization và training. Chỉ cần chạy theo thứ tự; không cần source Dataset. Code được sinh từ modules chung để B1/B2 có cùng logic.
Không có dữ liệu nguồn hoặc model weights được nhúng sẵn ở đây.


In [ ]:
# ===== data.py: Đọc TSV, kiểm tra frozen manifest =====
"""Immutable input loading, conservative alignment and auditable duplicates."""
import csv
import hashlib
import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path

SCHEMAS = {
    'nn': dict(target='Compound', parts=['Mod', 'Head'], labels=['ModAvg', 'HeadAvg'], std=['ModStd', 'HeadStd']),
    'pv': dict(target='ParticleVerb', parts=['Base', 'Particle'], labels=['Avg'], std=['Std']),
}

def dump(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def find_file(root, name):
    root = Path(root)
    matches = sorted(root.rglob(name)) if root.is_dir() else ([root] if root.name == name else [])
    if len(matches) != 1:
        raise ValueError(f'Expected one {name} under {root}; found {len(matches)}: {matches}. Set DATA_ROOT to a narrower directory.')
    return matches[0]

# Explicit conservative morphology; unlisted irregular forms require manual review.
IRREGULAR = {'take': ['took','taken'], 'give':['gave','given'], 'go':['went','gone'],
             'get':['got','gotten'], 'come':['came'], 'bring':['brought'], 'break':['broke','broken'],
             'run':['ran'], 'put':['put'], 'set':['set'], 'make':['made'], 'keep':['kept'],
             'hold':['held'], 'stand':['stood'], 'fall':['fell','fallen'], 'throw':['threw','thrown'],
             'leave':['left'], 'turn':['turned'], 'cut':['cut'], 'lay':['laid'], 'lie':['lay','lain'],
             'man':['men'], 'woman':['women'], 'child':['children'], 'foot':['feet'], 'tooth':['teeth']}

def forms(word, inflect=True):
    w = word.lower(); result = {w}
    if not inflect: return result
    result.update(IRREGULAR.get(w, []))
    result.update([w+'s', w+'es', w+'ed', w+'ing'])
    if w.endswith('e'): result.update([w+'d', w[:-1]+'ing'])
    if w.endswith('y') and len(w)>1 and w[-2] not in 'aeiou': result.update([w[:-1]+'ies',w[:-1]+'ied'])
    if len(w)>2 and w[-1] not in 'aeiouwxy' and w[-2] in 'aeiou' and w[-3] not in 'aeiou':
        result.update([w+w[-1]+'ed', w+w[-1]+'ing'])
    return result

def occurrences(text, word, inflect=True):
    # IGNORECASE works on the original text: offsets never refer to a normalized copy.
    options = sorted(forms(word, inflect), key=lambda x:(-len(x),x))
    return [(m.start(),m.end()) for m in re.finditer(r'(?<!\w)(?:'+ '|'.join(map(re.escape,options))+r')(?!\w)',text,re.I)]

def align(record):
    text = record['context']; task = record['task']
    a = occurrences(text, record['part1'])
    b = occurrences(text, record['part2'], inflect=task=='nn')
    pairs = []
    for x in a:
        for y in b:
            if x[1] > y[0]: continue
            gap = text[x[1]:y[0]]
            if task=='nn' and re.search(r'[^\s\-–—]',gap): continue
            pairs.append([list(x),list(y)])
    if len(pairs) != 1:
        return dict(spans=None, alignment_status='ambiguous' if pairs else 'missing', candidates=pairs)
    spans=pairs[0]
    exact=all(text[s:e].lower()==record[k].lower() for (s,e),k in zip(spans,['part1','part2']))
    return dict(spans=spans, alignment_status='exact' if exact else 'inflection', candidates=pairs)

def load_records(path, task, labeled=True, overrides=None):
    schema=SCHEMAS[task]; records=[]; seen=set(); overrides=overrides or {}
    with open(path, encoding='utf-8-sig', newline='') as f:
        reader=csv.DictReader(f, delimiter='\t')
        required=['ContextID', schema['target'], *schema['parts'], 'Context']
        if labeled: required += schema['labels']+schema['std']
        missing=set(required)-set(reader.fieldnames or [])
        if missing: raise ValueError(f'Missing columns: {sorted(missing)}')
        for line,row in enumerate(reader,2):
            if None in row or any(v is None for v in row.values()): raise ValueError(f'TSV field count mismatch at line {line}')
            if any(not row.get(k,'').strip() for k in ['ContextID',schema['target'],*schema['parts'],'Context']):
                raise ValueError(f'Missing identity/text at line {line}')
            cid=row['ContextID']
            if cid in seen: raise ValueError(f'Duplicate ID {cid}')
            seen.add(cid)
            def numbers(cols):
                vals=[]
                for c in cols:
                    s=row.get(c,'').strip()
                    v=float(s) if s else None
                    if v is not None and not math.isfinite(v): raise ValueError(f'Nonfinite {c} at {cid}')
                    vals.append(v)
                return vals
            r=dict(context_id=cid,task=task,target=row[schema['target']],context=row['Context'],
                   part1=row[schema['parts'][0]],part2=row[schema['parts'][1]],
                   labels=numbers(schema['labels']),annotation_std=numbers(schema['std']))
            if any(v is not None and v<0 for v in r['annotation_std']): raise ValueError(f'Negative Std at {cid}')
            r.update(align(r))
            if cid in overrides:
                spans=overrides[cid]
                if len(spans)!=2 or any(len(s)!=2 or not 0<=s[0]<s[1]<=len(r['context']) for s in spans):
                    raise ValueError(f'Invalid manual spans: {cid}')
                if spans[0][1]>spans[1][0]: raise ValueError(f'Overlapping/reversed spans: {cid}')
                r.update(spans=spans,alignment_status='manual')
            records.append(r)
    if not records: raise ValueError('Empty dataset')
    # B1 must be invariant across occurrences of a target.
    parts=defaultdict(set)
    for r in records: parts[r['target']].add((r['part1'].lower(),r['part2'].lower()))
    bad={k:sorted(v) for k,v in parts.items() if len(v)>1}
    if bad: raise ValueError(f'Inconsistent canonical constituents; resolve explicitly: {bad}')
    return records

def normalized_context(text):
    return ' '.join(re.findall(r'\w+',text.casefold()))

def duplicates(records, threshold=.9):
    """Exact and token-set Jaccard near duplicates. Inverted index avoids all-pairs text scans."""
    exact=[]; near=[]; index=defaultdict(list); sets=[]; texts=[]
    for i,r in enumerate(records):
        text=normalized_context(r['context']); tokens=set(text.split()); candidates=set()
        for t in tokens: candidates.update(index[t])
        for j in sorted(candidates):
            if texts[j]==text: exact.append([records[j]['context_id'],r['context_id']])
            elif len(tokens & sets[j])/max(1,len(tokens | sets[j])) >= threshold:
                near.append([records[j]['context_id'],r['context_id']])
        for t in tokens: index[t].append(i)
        sets.append(tokens); texts.append(text)
    return dict(exact=exact,near=near,near_policy=f'token-set Jaccard >= {threshold}; candidate review, not semantic equivalence')

def audit(records):
    import numpy as np
    def stats(values):
        v=np.array([x for x in values if x is not None],dtype=float)
        return dict(n=len(v),missing=len(values)-len(v),min=float(v.min()) if len(v) else None,
                    max=float(v.max()) if len(v) else None,mean=float(v.mean()) if len(v) else None,
                    sd=float(v.std()) if len(v) else None)
    counts=Counter(r['target'] for r in records)
    return dict(rows=len(records),targets=len(counts),contexts_per_target=dict(sorted(counts.items())),
                alignment=dict(Counter(r['alignment_status'] for r in records)),
                labels={k:stats([r['labels'][i] for r in records]) for i,k in enumerate(SCHEMAS[records[0]['task']]['labels'])},
                annotation_std={k:stats([r['annotation_std'][i] for r in records]) for i,k in enumerate(SCHEMAS[records[0]['task']]['std'])},
                label_scale='Official endpoints/direction unverified; observed distribution only; no clipping')


# ===== splits.py: Đọc TSV, kiểm tra frozen manifest =====
"""Target-disjoint splits with exact-duplicate connected components."""
from collections import defaultdict
import hashlib
import json
import random
import numpy as np


def fingerprint(records):
    return hashlib.sha256(json.dumps(records,sort_keys=True,ensure_ascii=False).encode()).hexdigest()

def make_splits(records, seed=42):
    targets=sorted({r['target'] for r in records})
    if len(targets)<7: raise ValueError('At least 7 targets required for a meaningful 70/15/15 split')
    parent={t:t for t in targets}
    def root(t):
        while parent[t]!=t: t=parent[t]
        return t
    byid={r['context_id']:r for r in records}
    dup=duplicates(records)
    # Keep exact duplicate contexts in one partition even if annotated for different targets.
    for a,b in dup['exact']:
        x,y=root(byid[a]['target']),root(byid[b]['target'])
        parent[max(x,y)]=min(x,y)
    groups=defaultdict(list)
    for t in targets: groups[root(t)].append(t)
    if len(groups)<3: raise ValueError('Exact-duplicate components leave fewer than 3 independent groups')
    means={}
    for t in targets:
        values=[v for r in records if r['target']==t for v in r['labels'] if v is not None]
        means[t]=float(np.mean(values)) if values else 0.
    # Sort by target-mean label, then randomize within short strata before weighted assignment.
    ordered=sorted(groups.values(),key=lambda g:(np.mean([means[t] for t in g]),g))
    rng=random.Random(seed); shuffled=[]
    for i in range(0,len(ordered),10):
        block=ordered[i:i+10]; rng.shuffle(block); shuffled.extend(block)
    desired=np.array([.7,.15,.15])*len(targets); counts=np.zeros(3); assigned={}
    for g in shuffled:
        scores=(desired-counts)/np.maximum(desired,1)
        k=int(np.argmax(scores)); counts[k]+=len(g)
        for t in g: assigned[t]=['train','dev','test'][k]
    partitions={s:[r['context_id'] for r in records if assigned[r['target']]==s] for s in ['train','dev','test']}
    if any(not x for x in partitions.values()): raise ValueError('Empty split: insufficient independent components')
    manifest=dict(version=1,seed=seed,task=records[0]['task'],records_sha256=fingerprint(records),
                  partitions=partitions,target_partition=assigned,policy='target-grouped; label-mean strata; exact-context components',
                  duplicate_report=dup)
    validate_splits(records,manifest)
    manifest['overlap']=overlap_report(records,manifest)
    return manifest

def validate_splits(records,manifest):
    if fingerprint(records)!=manifest['records_sha256']: raise ValueError('Manifest does not match normalized records')
    ids=[r['context_id'] for r in records]; flat=[x for v in manifest['partitions'].values() for x in v]
    if len(flat)!=len(set(flat)) or set(flat)!=set(ids): raise ValueError('Split ID coverage/overlap failure')
    if manifest.get('protocol')=='paired_seen_unseen':
        return validate_diagnostic(records,manifest)
    byid={r['context_id']:r for r in records}; seen={}
    for split,items in manifest['partitions'].items():
        for i in items:
            t=byid[i]['target']
            if t in seen and seen[t]!=split: raise ValueError('Target leakage')
            seen[t]=split
    loc={i:s for s,v in manifest['partitions'].items() for i in v}
    for a,b in duplicates(records)['exact']:
        if loc[a]!=loc[b]: raise ValueError('Exact duplicate context leakage')
    return True

def overlap_report(records,manifest):
    loc={i:s for s,v in manifest['partitions'].items() for i in v}; out={}
    for a,b in [('train','dev'),('train','test'),('dev','test')]:
        sets=[]
        for s in [a,b]: sets.append({r[p].casefold() for r in records if loc[r['context_id']]==s for p in ['part1','part2']})
        out[a+'_'+b]=dict(constituent_overlap=sorted(sets[0]&sets[1]))
    out['near_duplicates_cross_split']=[p for p in manifest['duplicate_report']['near'] if loc[p[0]]!=loc[p[1]]]
    return out


def seen_unseen_design(records, manifest, support_count=1, seed=42):
    """Additional RQ2 diagnostic: same Q, equal row budgets, independent dev targets."""
    validate_splits(records,manifest); rng=random.Random(seed)
    byid={r['context_id']:r for r in records}; groups=defaultdict(list)
    for i in manifest['partitions']['test']: groups[byid[i]['target']].append(i)
    support=[]; query=[]; excluded=[]
    # Do not separate exact/near duplicate contexts between support and query.
    forbidden={frozenset(p) for k in ['exact','near'] for p in manifest['duplicate_report'][k]}
    for t,ids in sorted(groups.items()):
        ids=sorted(ids); rng.shuffle(ids)
        s=ids[:support_count]; q=[i for i in ids[support_count:] if all(frozenset([i,j]) not in forbidden for j in s)]
        if len(ids)>support_count and q: support+=s; query+=q
        else: excluded.append(t)
    controls=list(manifest['partitions']['train']); rng.shuffle(controls)
    if not support or len(controls)<=len(support): raise ValueError('Insufficient support/query/control examples')
    control=controls[:len(support)]; base=controls[len(support):]
    return dict(query=query,support=support,seen_train=base+support,unseen_train=base+control,
                dev=manifest['partitions']['dev'],excluded_targets=excluded,support_count=support_count,
                confounds=['Control lexical/domain/label distribution may differ','Task-unseen is not pretraining-unseen'],
                source_manifest_sha256=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest())


def diagnostic_manifests(records, main_manifest, design):
    """Build explicit paired training pools; no hidden relaxation of main split validation."""
    validate_splits(records,main_manifest)
    pools={}
    for condition in ['seen','unseen']:
        partitions=dict(train=design[condition+'_train'],dev=design['dev'],test=design['query'])
        chosen={i for ids in partitions.values() for i in ids}
        pool=[r for r in records if r['context_id'] in chosen]
        manifest=dict(protocol='paired_seen_unseen',condition=condition,partitions=partitions,
                      records_sha256=fingerprint(pool),task=pool[0]['task'],design=design,
                      seed=main_manifest['seed'])
        validate_splits(pool,manifest); pools[condition]=(pool,manifest)
    if len(pools['seen'][1]['partitions']['train'])!=len(pools['unseen'][1]['partitions']['train']):
        raise ValueError('Unequal paired training budget')
    return pools


def validate_diagnostic(records,manifest):
    byid={r['context_id']:r for r in records}; parts=manifest['partitions']; d=manifest['design']
    expected=dict(train=d[manifest['condition']+'_train'],dev=d['dev'],test=d['query'])
    if parts!=expected: raise ValueError('Diagnostic partitions differ from design')
    sets={s:{byid[i]['target'] for i in ids} for s,ids in parts.items()}
    if sets['dev'] & (sets['train']|sets['test']): raise ValueError('Dev target leakage')
    if manifest['condition']=='unseen' and sets['train'] & sets['test']: raise ValueError('Unseen target leakage')
    if manifest['condition']=='seen' and not sets['test'] <= sets['train']: raise ValueError('Seen queries lack support')
    loc={i:s for s,v in parts.items() for i in v}
    dup=duplicates(records)
    for kind in ['exact','near']:
        for a,b in dup[kind]:
            if loc[a]!=loc[b]: raise ValueError('Diagnostic duplicate context leakage')
    return True


# ===== kaggle_input.py: Đọc TSV, kiểm tra frozen manifest =====
"""Read the frozen data bundle on Kaggle. Never generate or change split membership."""
import hashlib
import json
from pathlib import Path
import zipfile



def locate_frozen_task(data_root, task, working_root):
    if task not in ['nn', 'pv']:
        raise ValueError('TASK must be nn or pv')
    data_root = Path(data_root)
    if not data_root.is_dir():
        raise FileNotFoundError(f'DATA_ROOT does not exist: {data_root}')
    manifest_name = f'split_manifest_{task}.json'
    raw_name = f'en-{task}-train.tsv'
    matches = sorted({p.parent for p in data_root.rglob(manifest_name)
                      if (p.parent / raw_name).is_file() and (p.parent / 'dataset_info.json').is_file()})
    if not matches:
        archives = sorted(data_root.rglob('cosense-kaggle-data.zip'))
        if len(archives) != 1:
            raise ValueError(f'Need one frozen data bundle, not only a raw TSV. '
                             f'Attach cosense-kaggle-data.zip or narrow DATA_ROOT. Found ZIPs: {archives}')
        archive = archives[0]
        checksum = hashlib.sha256(archive.read_bytes()).hexdigest()[:12]
        extracted = Path(working_root) / f'frozen_data_{checksum}'
        extracted.mkdir(parents=True, exist_ok=True)
        expected = {'README.md', 'dataset_info.json', 'en-nn-train.tsv', 'en-pv-train.tsv',
                    'split_manifest_nn.json', 'split_manifest_pv.json'}
        with zipfile.ZipFile(archive) as z:
            if len(z.namelist()) != len(set(z.namelist())) or set(z.namelist()) != expected:
                raise ValueError('Unexpected/missing data ZIP members')
            if sum(i.file_size for i in z.infolist()) > 100_000_000:
                raise ValueError('Data ZIP exceeds 100 MB uncompressed')
            for name in sorted(expected):
                (extracted / name).write_bytes(z.read(name))
        matches = [extracted]
    if len(matches) != 1:
        raise ValueError(f'Multiple matching frozen datasets. Set DATA_ROOT explicitly: {matches}')
    return matches[0]


def load_frozen_task(data_root, task, working_root, split_seed=42):
    folder = locate_frozen_task(data_root, task, working_root)
    raw_name = f'en-{task}-train.tsv'; manifest_name = f'split_manifest_{task}.json'
    info = json.loads((folder / 'dataset_info.json').read_text(encoding='utf-8-sig'))
    manifest = json.loads((folder / manifest_name).read_text(encoding='utf-8-sig'))
    if info.get('bundle_version') != 1 or info.get('split_seed') != split_seed:
        raise ValueError('Data bundle version/split seed mismatch')
    for name in [raw_name, manifest_name]:
        if digest(folder / name) != info.get('files_sha256', {}).get(name):
            raise ValueError(f'Frozen file checksum mismatch: {name}')
    if manifest.get('task') != task or manifest.get('seed') != split_seed:
        raise ValueError('Manifest task/seed mismatch')
    if manifest.get('source_file') != raw_name or manifest.get('source_sha256') != digest(folder / raw_name):
        raise ValueError('Manifest source identity mismatch')
    if set(manifest.get('partitions', {})) != {'train', 'dev', 'test'} or manifest.get('protocol') == 'paired_seen_unseen':
        raise ValueError('Expected the main target-disjoint train/dev/test manifest')
    records = load_records(folder / raw_name, task)
    validate_splits(records, manifest)
    provenance = dict(source_sha256=manifest['source_sha256'], manifest_sha256=digest(folder / manifest_name),
                      split_rows={s: len(ids) for s, ids in manifest['partitions'].items()},
                      split_seed=split_seed, task=task)
    return records, manifest, provenance


In [ ]:
# ===== metrics.py: Metrics, predictions và results bundles =====
"""Pairwise-complete evaluation; undefined correlations are None, never zero."""
from itertools import combinations
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


def correlation(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float); valid=np.isfinite(a)&np.isfinite(b)
    a=a[valid]; b=b[valid]
    reason=None
    if len(a)<2: reason='fewer_than_two'
    elif np.ptp(a)==0 or np.ptp(b)==0: reason='constant'
    return dict(value=None if reason else float(spearmanr(a,b).statistic),n=len(a),reason=reason)


def evaluate(frame, task):
    results={}; statistics=[]
    for output in SCHEMAS[task]['labels']:
        y=pd.to_numeric(frame['gold_'+output],errors='coerce'); p=pd.to_numeric(frame['pred_'+output],errors='coerce')
        valid=np.isfinite(y)&np.isfinite(p); f=frame.loc[valid].copy()
        f['_y']=y[valid]; f['_p']=p[valid]
        target_stats=[]; deltas=[]; pairs=0
        for target,g in f.groupby('target',sort=True):
            if len(g)<2: continue
            sy=0. if np.ptp(g['_y'])==0 else float(np.std(g['_y'],ddof=0))
            sp=0. if np.ptp(g['_p'])==0 else float(np.std(g['_p'],ddof=0))
            target_stats.append(dict(target=target,output=output,n=len(g),gold_sd=sy,pred_sd=sp))
            values=g[['_y','_p']].to_numpy()
            errors=[abs((a[0]-b[0])-(a[1]-b[1])) for a,b in combinations(values,2)]
            pairs+=len(errors); deltas.append(float(np.mean(errors)))
        statistics.extend(target_stats)
        results[output]=dict(spearman=correlation(y,p),rmse=float(np.sqrt(np.mean((y[valid]-p[valid])**2))) if valid.any() else None,
                             n_valid=int(valid.sum()),n_total=len(frame),
                             context_variance=correlation([s['gold_sd'] for s in target_stats],[s['pred_sd'] for s in target_stats]),
                             delta_mae_macro=float(np.mean(deltas)) if deltas else None,
                             delta_targets=len(deltas),delta_pairs=pairs)
    vals=[v['spearman']['value'] for v in results.values() if v['spearman']['value'] is not None]
    return dict(outputs=results,mean_valid_output_spearman=float(np.mean(vals)) if vals else None,
                valid_spearman_outputs=len(vals),policy=dict(ddof=0,ties='average ranks',missing='pairwise finite',
                singletons='excluded from SD and deltas',undefined='null with reason and coverage')),pd.DataFrame(statistics)


def compare_predictions(b1,b2,task,replicates=1000,seed=42):
    """Paired target-cluster bootstrap of macro target MAE gain (positive = B2 better)."""
    keys=['task','context_id','target']
    for col in ['seed','split','task']:
        if set(b1[col].unique())!=set(b2[col].unique()) or b1[col].nunique()!=1:
            raise ValueError('Paired identity mismatch: '+col)
    if set(b1['model'])!={'B1'} or set(b2['model'])!={'B2'}: raise ValueError('Expected B1 then B2')
    if b1.duplicated(keys).any() or b2.duplicated(keys).any(): raise ValueError('Duplicate prediction IDs')
    if set(map(tuple,b1[keys].to_numpy()))!=set(map(tuple,b2[keys].to_numpy())): raise ValueError('Prediction coverage differs')
    f=b1.merge(b2,on=keys,suffixes=('_b1','_b2'),validate='one_to_one'); out={}; rng=np.random.default_rng(seed)
    for o in SCHEMAS[task]['labels']:
        a=f['gold_'+o+'_b1'].to_numpy(float); b=f['gold_'+o+'_b2'].to_numpy(float)
        if not np.allclose(a,b,equal_nan=True): raise ValueError('Gold mismatch')
        gain=np.abs(a-f['pred_'+o+'_b1'])-np.abs(a-f['pred_'+o+'_b2'])
        f['context_gain_'+o]=gain
        means=f.assign(_gain=gain).groupby('target')['_gain'].mean().dropna().to_numpy()
        if len(means):
            boot=[float(rng.choice(means,len(means),replace=True).mean()) for _ in range(replicates)]
            out[o]=dict(macro_mae_gain=float(means.mean()),ci95=list(map(float,np.quantile(boot,[.025,.975]))),targets=len(means),replicates=replicates)
        else: out[o]=dict(macro_mae_gain=None,ci95=None,targets=0)
    return out,f


def error_groups(frame,task):
    """Descriptive RQ3 associations, no inferred semantic metadata."""
    f=frame.copy(); group_columns=['alignment_status']
    if 'fallback_reason' in f:
        f['used_fallback']=f['fallback_reason'].fillna('').astype(str).str.len()>0
        group_columns.append('used_fallback')
    if 'truncated' in f: group_columns.append('truncated')
    for col in ['context_length','subword_count']:
        if col in f:
            f[col+'_bin']=pd.qcut(f[col],q=4,duplicates='drop').astype(str)
            group_columns.append(col+'_bin')
    rows=[]
    for col in group_columns:
        for value,g in f.groupby(col,dropna=False):
            for o in SCHEMAS[task]['labels']:
                error=(g['gold_'+o]-g['pred_'+o]).abs()
                rows.append(dict(group=col,value=str(value),output=o,rows=len(g),valid=int(error.notna().sum()),
                                 targets=g['target'].nunique(),mae=float(error.mean()) if error.notna().any() else None))
    return pd.DataFrame(rows)


# ===== artifacts.py: Metrics, predictions và results bundles =====
"""Results bundle and ordered submission contracts."""
import csv
import io
import json
import math
from pathlib import Path
import zipfile
import pandas as pd


def prediction_frame(records,predictions,model,seed,split,logs=None):
    rows=[]
    for i,(r,p) in enumerate(zip(records,predictions,strict=True)):
        row={k:r[k] for k in ['context_id','task','target','context','part1','part2','alignment_status']}
        row.update(spans=json.dumps(r['spans']),model=model,seed=seed,split=split,context_length=len(r['context'].split()))
        for j,o in enumerate(SCHEMAS[r['task']]['labels']):
            row['gold_'+o]=r['labels'][j]; row['pred_'+o]=float(p[j]); row['annotation_std_'+o]=r['annotation_std'][j]
            row['absolute_error_'+o]=abs(p[j]-r['labels'][j]) if r['labels'][j] is not None else None
        if logs: row.update(logs[i])
        rows.append(row)
    return pd.DataFrame(rows)


def export_bundle(directory,frame,metadata,manifest,history=None):
    directory=Path(directory); directory.mkdir(parents=True,exist_ok=True)
    metrics,stats=evaluate(frame,metadata['task'])
    dump(directory/'run_metadata.json',dict(metadata,contract_version=1,output_order=SCHEMAS[metadata['task']]['labels']))
    dump(directory/'metrics.json',metrics); dump(directory/'split_manifest.json',manifest)
    frame.to_csv(directory/'predictions.tsv',sep='\t',index=False)
    frame.to_csv(directory/'error_analysis.tsv',sep='\t',index=False)
    stats.to_csv(directory/'target_statistics.tsv',sep='\t',index=False)
    error_groups(frame,metadata['task']).to_csv(directory/'error_groups.tsv',sep='\t',index=False)
    pd.DataFrame(history or [],columns=['epoch','train_loss','dev_spearman','dev_rmse','seconds']).to_csv(directory/'training_history.csv',index=False)
    names=['run_metadata.json','metrics.json','predictions.tsv','target_statistics.tsv','error_analysis.tsv','training_history.csv','split_manifest.json','error_groups.tsv']
    with zipfile.ZipFile(directory/'results_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
        for name in names: z.write(directory/name,name)
    return metrics


def read_bundle(source):
    with zipfile.ZipFile(source) as z:
        needed=['run_metadata.json','metrics.json','predictions.tsv','error_analysis.tsv']
        if any(n not in z.namelist() for n in needed): raise ValueError('Incomplete results bundle')
        if sum(i.file_size for i in z.infolist())>200_000_000: raise ValueError('Bundle exceeds 200 MB uncompressed')
        meta=json.loads(z.read('run_metadata.json')); metrics=json.loads(z.read('metrics.json'))
        frame=pd.read_csv(io.BytesIO(z.read('predictions.tsv')),sep='\t',dtype={'context_id':str})
        if meta.get('output_order')!=SCHEMAS[meta['task']]['labels']: raise ValueError('Output order metadata mismatch')
        if frame['context_id'].duplicated().any(): raise ValueError('Duplicate IDs in bundle')
        return meta,metrics,frame


def write_submission(path,records,frame,metadata):
    task=records[0]['task']; labels=SCHEMAS[task]['labels']
    if metadata.get('task')!=task or metadata.get('output_order')!=labels: raise ValueError('Task/output order metadata mismatch')
    ids=[r['context_id'] for r in records]; frame=frame.copy(); frame['context_id']=frame['context_id'].astype(str)
    if frame['context_id'].duplicated().any() or set(ids)!=set(frame['context_id']): raise ValueError('Submission ID coverage mismatch')
    ordered=frame.set_index('context_id').loc[ids]
    values=ordered[['pred_'+o for o in labels]].to_numpy(float)
    if not all(math.isfinite(v) for row in values for v in row): raise ValueError('Nonfinite predictions')
    with open(path,'w',encoding='utf-8',newline='') as f:
        w=csv.writer(f,delimiter='\t',lineterminator='\n')
        for cid,row in zip(ids,values): w.writerow([cid,*[format(v,'.9g') for v in row]])
    validate_submission(path,ids,task)


def validate_submission(path,expected_ids,task):
    rows=list(csv.reader(open(path,encoding='utf-8'),delimiter='\t'))
    if len(rows)!=len(expected_ids): raise ValueError('Row count mismatch')
    if [r[0] for r in rows]!=list(expected_ids) or len(set(expected_ids))!=len(expected_ids): raise ValueError('ID order/coverage mismatch')
    for r in rows:
        if len(r)!=1+len(SCHEMAS[task]['labels']): raise ValueError('Column count mismatch')
        if not all(math.isfinite(float(v)) for v in r[1:]): raise ValueError('Nonfinite prediction')
    return True


def submission_zip(path,files):
    if not files: raise ValueError('No settings selected')
    allowed={'en-nn-pred.tsv','en-pv-pred.tsv'}
    if any(Path(f).name not in allowed for f in files): raise ValueError('Unexpected submission filename')
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        for f in files: z.write(f,Path(f).name)


# ===== published_outputs.py: Metrics, predictions và results bundles =====
"""Publish verified run artifacts as visible files in Kaggle's output root."""
import json
from pathlib import Path
import shutil
import zipfile

import pandas as pd


def publish_run_artifacts(run_root, publish_root, model, task, seed, smoke=False):
    run_root = Path(run_root); publish_root = Path(publish_root)
    required = ['best.pt', 'resume.pt', 'config.json', 'run_metadata.json', 'split_manifest.json',
                'history.json', 'encoder_config/config.json', 'tokenizer/tokenizer.json',
                'tokenizer/tokenizer_config.json', 'dev/metrics.json', 'dev/predictions.tsv',
                'dev/training_history.csv', 'dev/results_bundle.zip']
    missing = [name for name in required if not (run_root/name).is_file() or (run_root/name).stat().st_size == 0]
    if missing:
        raise FileNotFoundError(f'Run has no complete artifacts; training/export is not finished. Missing: {missing}')
    metadata, metrics, frame = read_bundle(run_root/'dev/results_bundle.zip')
    if metadata.get('model') != model or metadata.get('task') != task or metadata.get('seed') != seed:
        raise ValueError('Cannot publish a different model/task/seed')
    if metadata.get('config', {}).get('smoke') != smoke:
        raise ValueError('Cannot publish a different smoke/full run')
    if model not in ['B1','B2'] or task not in ['nn','pv'] or not isinstance(seed,int):
        raise ValueError('Unsupported output identity')
    if frame.empty:
        raise ValueError('Predictions are empty')
    publish_root.mkdir(parents=True, exist_ok=True)
    tag = f'{task.upper()}_{model}_seed{seed}_{"smoke" if smoke else "full"}'
    results = publish_root/f'{tag}_results.zip'
    model_zip = publish_root/f'{tag}_model.zip'
    print('EXPORT: results ZIP ->', results, flush=True)
    shutil.copyfile(run_root/'dev/results_bundle.zip', results)
    # Stored ZIP avoids spending minutes recompressing floating-point weights.
    # Resume checkpoint remains in the run folder; the model ZIP is for inference.
    print('EXPORT: model ZIP ->', model_zip, flush=True)
    temporary = model_zip.with_suffix('.zip.tmp')
    names = ['best.pt','config.json','run_metadata.json','split_manifest.json','history.json']
    names += [p.relative_to(run_root).as_posix() for folder in ['tokenizer','encoder_config']
              for p in (run_root/folder).rglob('*') if p.is_file()]
    for name in ['data_provenance.json','progress.json']:
        if (run_root/name).is_file(): names.append(name)
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_STORED) as z:
        for name in sorted(set(names)):
            z.write(run_root/name,name)
    temporary.replace(model_zip)
    table_path = publish_root/f'{tag}_metrics.csv'
    pd.DataFrame([dict(model=model,task=task,seed=seed,smoke=smoke,split='dev',output=output,
                       n=v['n_valid'],spearman=v['spearman']['value'],rmse=v['rmse'],
                       variance_correlation=v['context_variance']['value'],delta_mae=v['delta_mae_macro'])
                  for output,v in metrics['outputs'].items()]).to_csv(table_path,index=False)
    history_path = publish_root/f'{tag}_history.csv'
    shutil.copyfile(run_root/'dev/training_history.csv',history_path)
    paths = [results,model_zip,table_path,history_path]
    inventory = dict(status='OUTPUTS_READY',model=model,task=task,seed=seed,smoke=smoke,
                     run_directory=str(run_root),resume_checkpoint=str(run_root/'resume.pt'),
                     prediction_rows=len(frame),
                     files=[dict(name=p.name,bytes=p.stat().st_size) for p in paths])
    for item in inventory['files']:
        if item['bytes'] == 0: raise ValueError(f'Empty published artifact: {item["name"]}')
    manifest_path = publish_root/f'{tag}_ARTIFACTS.json'
    dump(manifest_path,inventory)
    print('OUTPUTS_READY:',manifest_path,flush=True)
    for item in inventory['files']: print(item['name'],item['bytes'],'bytes',flush=True)
    return manifest_path


def verify_published_artifacts(manifest_path):
    manifest_path = Path(manifest_path)
    inventory = json.loads(manifest_path.read_text(encoding='utf-8-sig'))
    if inventory.get('status') != 'OUTPUTS_READY' or not inventory.get('files'):
        raise ValueError('Publication has no completed output inventory')
    for item in inventory['files']:
        name=item['name']
        if Path(name).name != name: raise ValueError('Unsafe artifact name')
        path=manifest_path.parent/name
        if not path.is_file() or path.stat().st_size != item['bytes'] or item['bytes'] <= 0:
            raise FileNotFoundError(f'Published artifact missing/changed: {path}')
    return inventory


In [ ]:
# ===== tokenization.py: Tokenization, model và training với progress logs =====
"""Fast-tokenizer span masks and target-preserving token windows."""
import numpy as np


def offset_mask(offsets, span):
    a,b=span
    return [int(e>s and s<b and e>a) for s,e in offsets]


def window_bounds(masks, budget, length):
    positions=[i for i in range(length) if any(m[i] for m in masks)]
    if not positions or any(not any(m) for m in masks): raise ValueError('Constituent has no subwords')
    lo,hi=min(positions),max(positions)+1
    if hi-lo>budget: raise ValueError('Target constituents do not fit in one window')
    start=max(0,min(lo-(budget-(hi-lo))//2,length-budget)); end=min(length,start+budget)
    return start,end


def expression(record):
    a=record['part1'].lower().strip(); b=record['part2'].lower().strip()
    return a+' '+b,[[0,len(a)],[len(a)+1,len(a)+1+len(b)]]


def encode(record,tokenizer,model='B2',max_length=128,fallback='expression'):
    if not tokenizer.is_fast: raise ValueError('Fast tokenizer required')
    if fallback not in ['expression','error']: raise ValueError('Unknown fallback policy')
    text,spans=expression(record) if model=='B1' else (record['context'],record['spans'])
    reason=None
    def raw(text,spans):
        if spans is None: raise ValueError('Unresolved alignment: '+record['alignment_status'])
        enc=tokenizer(text,add_special_tokens=True,return_offsets_mapping=True,return_special_tokens_mask=True,truncation=False)
        special=enc['special_tokens_mask']
        content=[i for i,s in enumerate(special) if not s]
        offsets=[enc['offset_mapping'][i] for i in content]
        masks=[offset_mask(offsets,s) for s in spans]
        for span,mask in zip(spans,masks):
            selected=[o for o,m in zip(offsets,mask) if m]
            if not selected or selected[0][0]>span[0] or selected[-1][1]<span[1]: raise ValueError('Incomplete constituent tokens')
        budget=max_length-sum(special)
        if budget<2: raise ValueError('max_length too small')
        start,end=window_bounds(masks,budget,len(content))
        selected=set(content[start:end]); keep=[i for i,s in enumerate(special) if s or i in selected]
        built=[enc['input_ids'][i] for i in keep]
        lookup={v:i for i,v in enumerate(content)}
        full_masks=[[0 if special[i] else mask[lookup[i]] for i in keep] for mask in masks]
        context_mask=[int(not special[i] and not a and not b) for i,a,b in zip(keep,*full_masks)]
        n=len(built); pad=max_length-n
        return dict(input_ids=built+[tokenizer.pad_token_id]*pad,attention_mask=[1]*n+[0]*pad,
                    part1_mask=full_masks[0]+[0]*pad,part2_mask=full_masks[1]+[0]*pad,
                    context_mask=context_mask+[0]*pad,truncated=start>0 or end<len(content),
                    subword_count=len(content),window_start=start,window_end=end)
    try: result=raw(text,spans)
    except ValueError as e:
        if model=='B1' or fallback=='error': raise
        reason=str(e); text,spans=expression(record); result=raw(text,spans)
        # An expression fallback has no sentence context to pool in B3.
        result['context_mask']=[0]*max_length
    result['fallback_reason']=reason
    return result


class EncodedDataset:
    def __init__(self,records,tokenizer,config,model):
        import torch
        self.items=[]; self.logs=[]
        for r in records:
            e=encode(r,tokenizer,model,config['max_length'],config.get('fallback','expression'))
            self.logs.append({k:e[k] for k in ['truncated','subword_count','window_start','window_end','fallback_reason']})
            self.logs[-1]['source_context_subwords']=len(tokenizer(r['context'],add_special_tokens=False)['input_ids'])
            item={k:torch.tensor(e[k],dtype=torch.long) for k in ['input_ids','attention_mask','part1_mask','part2_mask','context_mask']}
            item['labels']=torch.tensor([float('nan') if v is None else v for v in r['labels']],dtype=torch.float32)
            self.items.append(item)
    def __len__(self): return len(self.items)
    def __getitem__(self,i): return self.items[i]


# ===== model.py: Tokenization, model và training với progress logs =====
"""Same two-constituent head capacity for B1/B2; optional B3 context pooling."""
import torch
from torch import nn
from transformers import AutoModel


class Regressor(nn.Module):
    def __init__(self,backbone,outputs,variant='B2',revision='main',layer=-1,encoder=None):
        super().__init__()
        self.encoder=encoder if encoder is not None else AutoModel.from_pretrained(backbone,revision=revision)
        self.variant=variant; self.layer=layer
        width=self.encoder.config.hidden_size*(3 if variant=='B3' else 2)
        self.head=nn.Sequential(nn.Dropout(.1),nn.Linear(width,outputs))
    @staticmethod
    def pool(hidden,mask):
        mask=mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden*mask).sum(1)/mask.sum(1).clamp_min(1)
    def forward(self,input_ids,attention_mask,part1_mask,part2_mask,context_mask,labels=None):
        out=self.encoder(input_ids=input_ids,attention_mask=attention_mask,output_hidden_states=self.layer!=-1)
        hidden=out.last_hidden_state if self.layer==-1 else out.hidden_states[self.layer]
        pooled=[self.pool(hidden,part1_mask),self.pool(hidden,part2_mask)]
        if self.variant=='B3': pooled.append(self.pool(hidden,context_mask))
        return self.head(torch.cat(pooled,dim=-1))


def masked_mse(pred,labels):
    valid=torch.isfinite(labels); losses=[]
    # Equal weight per valid output, even when label missingness differs.
    for i in range(labels.shape[1]):
        mask=valid[:,i]
        if mask.any(): losses.append(((pred[mask,i]-labels[mask,i])**2).mean())
    if not losses: return pred.sum()*0
    return torch.stack(losses).mean()


# ===== training.py: Tokenization, model và training với progress logs =====
"""Kaggle training and epoch-boundary resume; CPU requires explicit smoke mode."""
import copy
import importlib.metadata
import json
import math
import os
from pathlib import Path
import platform
import random
import time
import numpy as np


def report_progress(directory, stage, **details):
    """Flush live logs and a small status file without affecting training/RNG state."""
    status = dict(stage=stage, **details)
    dump(Path(directory)/'progress.json', status)
    print(f'[{stage}] ' + json.dumps(details, ensure_ascii=False, allow_nan=False), flush=True)


def seed_all(seed):
    import torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def runtime_metadata(config,task,variant):
    import torch
    packages={}
    for name in ['torch','transformers','tokenizers','numpy','pandas','scipy']:
        try: packages[name]=importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: packages[name]=None
    return dict(task=task,model=variant,seed=config['seed'],output_order=SCHEMAS[task]['labels'],config=config,packages=packages,
                python=platform.python_version(),device=torch.cuda.get_device_name() if torch.cuda.is_available() else 'cpu',
                resume_policy='epoch boundary; RNG/optimizer/scheduler/scaler/shuffle generator restored; cross-device bitwise equivalence not guaranteed')


def run_b0(records,manifest,output,seed=42,evaluate_test=False):
    validate_splits(records,manifest)
    train=[r for r in records if r['context_id'] in set(manifest['partitions']['train'])]
    matrix=np.array([[np.nan if v is None else v for v in r['labels']] for r in train])
    if np.any(np.all(np.isnan(matrix),axis=0)): raise ValueError('Training output has no labels')
    means=np.nanmean(matrix,axis=0)
    for split in ['dev']+(['test'] if evaluate_test else []):
        subset=[r for r in records if r['context_id'] in set(manifest['partitions'][split])]
        frame=prediction_frame(subset,np.tile(means,(len(subset),1)),'B0',seed,split)
        export_bundle(Path(output)/split,frame,dict(task=records[0]['task'],model='B0',seed=seed,training_mean=means.tolist()),manifest)
    return means


def predict(model,dataset,batch_size,device):
    import torch
    from torch.utils.data import DataLoader
    model.eval(); preds=[]
    # Cache identical B1 encodings: guarantee type-invariant outputs even across batch kernels.
    inverse=None
    if model.variant=='B1':
        unique=[]; lookup={}; inverse=[]
        for item in dataset:
            key=tuple(item['input_ids'].tolist())+tuple(item['attention_mask'].tolist())+tuple(item['part1_mask'].tolist())+tuple(item['part2_mask'].tolist())
            if key not in lookup: lookup[key]=len(unique); unique.append(item)
            inverse.append(lookup[key])
        dataset=unique
    with torch.no_grad():
        for batch in DataLoader(dataset,batch_size=batch_size,shuffle=False):
            preds.append(model(**{k:v.to(device) for k,v in batch.items()}).float().cpu().numpy())
    result=np.concatenate(preds)
    return result[inverse] if inverse is not None else result


def train(records,manifest,config,variant,output,resume=None,final=False,evaluate_test=False,max_epochs_this_invocation=None):
    import torch
    from torch.utils.data import DataLoader
    from transformers import AutoTokenizer,AutoModel,AutoConfig
    config=copy.deepcopy(config); output=Path(output); output.mkdir(parents=True,exist_ok=True)
    if variant not in ['B1','B2','B3']: raise ValueError('Unknown variant')
    if not final: validate_splits(records,manifest)
    if not torch.cuda.is_available() and not (config.get('smoke') and config.get('allow_cpu')):
        raise RuntimeError('Neural training needs CUDA. For a small CPU test set smoke=true and allow_cpu=true explicitly.')
    if final and (not config.get('recipe_locked') or config.get('smoke')): raise ValueError('Final training requires a locked nonsmoke recipe')
    if config['batch_size']<1 or config['accumulation']<1 or config['epochs']<1: raise ValueError('Invalid training budget')
    seed_all(config['seed']); device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    report_progress(output,'setup',model=variant,task=records[0]['task'],device=str(device),
                    smoke=config.get('smoke',False),output=str(output),resume=bool(resume))
    resume_state=torch.load(resume,map_location='cpu',weights_only=False) if resume else None
    if resume_state:
        expected=dict(config=config,variant=variant,manifest=manifest,final=final)
        if resume_state['identity']!=expected: raise ValueError('Resume config/split/variant mismatch; restore the saved config exactly')
        snapshot=Path(resume).parent
        tokenizer=AutoTokenizer.from_pretrained(snapshot/'tokenizer',use_fast=True,local_files_only=True)
        encoder=AutoModel.from_config(AutoConfig.from_pretrained(snapshot/'encoder_config',local_files_only=True))
    else:
        tokenizer=AutoTokenizer.from_pretrained(config['backbone'],revision=config.get('revision','main'),use_fast=True)
        encoder=None
    model=Regressor(config['backbone'],len(SCHEMAS[records[0]['task']]['labels']),variant,
                    config.get('revision','main'),config.get('layer',-1),encoder=encoder).to(device)
    if config['max_length']>getattr(model.encoder.config,'max_position_embeddings',config['max_length']): raise ValueError('max_length exceeds backbone positions')
    tokenizer.save_pretrained(output/'tokenizer'); model.encoder.config.save_pretrained(output/'encoder_config')
    partitions={'train':[r['context_id'] for r in records]} if final else manifest['partitions']
    subsets={s:[r for r in records if r['context_id'] in set(ids)] for s,ids in partitions.items()}
    if config.get('smoke'):
        subsets={s:v[:min(len(v),config.get('smoke_rows',16))] for s,v in subsets.items()}
    needed=['train'] if final else ['train','dev']+(['test'] if evaluate_test else [])
    datasets={}
    for split in needed:
        report_progress(output,'tokenize',split=split,rows=len(subsets[split]),max_length=config['max_length'])
        datasets[split]=EncodedDataset(subsets[split],tokenizer,config,variant)
    gen=torch.Generator().manual_seed(config['seed'])
    loader=DataLoader(datasets['train'],batch_size=config['batch_size'],shuffle=True,generator=gen,num_workers=0)
    optimizer=torch.optim.AdamW(model.parameters(),lr=config['lr'])
    steps_per_epoch=math.ceil(len(loader)/config['accumulation']); total=config['epochs']*steps_per_epoch
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda s:max(0.,1-s/max(1,total)))
    amp=device.type=='cuda'; dtype=torch.bfloat16 if amp and torch.cuda.is_bf16_supported() else torch.float16
    scaler=torch.amp.GradScaler('cuda',enabled=amp and dtype==torch.float16)
    best=None; best_rmse=float('inf'); stale=0; start=0; history=[]; step=0
    if resume_state:
        state=resume_state; model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']); scaler.load_state_dict(state['scaler'])
        start=state['epoch']+1; step=state['step']; best=state['best']; best_rmse=state['best_rmse']; stale=state['stale']; history=state['history']
        random.setstate(state['python_rng']); np.random.set_state(state['numpy_rng']); torch.set_rng_state(state['torch_rng'])
        gen.set_state(state['generator_rng'])
        if amp and state['cuda_rng'] is not None: torch.cuda.set_rng_state_all(state['cuda_rng'])
        if not (output/'best.pt').exists():
            source=Path(resume).parent/'best.pt'
            if source.exists():
                import shutil
                shutil.copy2(source,output/'best.pt')
            else: raise ValueError('Resume requires sibling best.pt')
    metadata=runtime_metadata(config,records[0]['task'],variant)
    metadata.update(trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),
                    model_revision_resolved=getattr(model.encoder.config,'_commit_hash',None),
                    tokenizer_revision_resolved=tokenizer.init_kwargs.get('_commit_hash'),
                    actual_split_ids={s:[r['context_id'] for r in subsets[s]] for s in needed},
                    preprocessing={s:dict(rows=len(d.logs),fallback=sum(x['fallback_reason'] is not None for x in d.logs),
                                         truncated=sum(x['truncated'] for x in d.logs)) for s,d in datasets.items()})
    dump(output/'run_metadata.json',metadata); dump(output/'config.json',config); dump(output/'split_manifest.json',manifest)
    started=time.perf_counter()
    end_epoch=config['epochs'] if max_epochs_this_invocation is None else min(config['epochs'],start+max_epochs_this_invocation)
    if end_epoch<=start and not resume_state: raise ValueError('Empty invocation epoch budget')
    log_every=max(1,int(config.get('log_every_batches',20)))
    report_progress(output,'ready',train_rows=len(subsets['train']),batches_per_epoch=len(loader),
                    optimizer_steps_per_epoch=steps_per_epoch,planned_epochs=config['epochs'],
                    effective_batch_size=config['batch_size']*config['accumulation'])
    for epoch in range(start,end_epoch):
        if not final and stale>=config['patience']:
            report_progress(output,'early_stop',completed_epochs=epoch,patience=config['patience'])
            break
        model.train(); optimizer.zero_grad(); losses=[]
        for i,batch in enumerate(loader):
            batch={k:v.to(device) for k,v in batch.items()}
            group_start=(i//config['accumulation'])*config['accumulation']
            divisor=min(config['accumulation'],len(loader)-group_start)
            with torch.autocast(device_type=device.type,dtype=dtype,enabled=amp):
                loss=masked_mse(model(**batch),batch['labels'])
            losses.append(float(loss.detach())); scaler.scale(loss/divisor).backward()
            if (i+1)%config['accumulation']==0 or i+1==len(loader):
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                scaler.step(optimizer); scaler.update(); scheduler.step(); optimizer.zero_grad(); step+=1
            if i==0 or (i+1)%log_every==0 or i+1==len(loader):
                report_progress(output,'train',epoch=epoch+1,epochs=config['epochs'],batch=i+1,
                                batches=len(loader),train_loss=float(np.mean(losses)),optimizer_step=step,
                                elapsed_seconds=round(time.perf_counter()-started,2))
        score=None; rmse=None
        if not final:
            report_progress(output,'dev',epoch=epoch+1,rows=len(subsets['dev']))
            preds=predict(model,datasets['dev'],config['batch_size'],device)
            frame=prediction_frame(subsets['dev'],preds,variant,config['seed'],'dev',datasets['dev'].logs)
            metrics,_=evaluate(frame,records[0]['task']); score=metrics['mean_valid_output_spearman']
            vals=[v['rmse'] for v in metrics['outputs'].values() if v['rmse'] is not None]
            rmse=float(np.mean(vals)) if vals else None
            if rmse is None: raise ValueError('No valid dev labels')
            report_progress(output,'dev_metrics',epoch=epoch+1,mean_valid_output_spearman=score,
                            outputs={o:dict(spearman=v['spearman']['value'],rmse=v['rmse'],
                                            variance_correlation=v['context_variance']['value'],n=v['n_valid'])
                                     for o,v in metrics['outputs'].items()})
        # Undefined rho: select lower dev RMSE until any defined rho exists. Never replace rho with zero.
        improved=final or (score is not None and (best is None or score>best)) or (score is None and best is None and rmse<best_rmse)
        if improved:
            best=score; best_rmse=rmse if rmse is not None else best_rmse; stale=0
            torch.save(dict(model=model.state_dict(),epoch=epoch),output/'best.pt')
        else: stale+=1
        history.append(dict(epoch=epoch,train_loss=float(np.mean(losses)),dev_spearman=score,dev_rmse=rmse,seconds=time.perf_counter()-started))
        state=dict(identity=dict(config=config,variant=variant,manifest=manifest,final=final),model=model.state_dict(),
                   optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),epoch=epoch,step=step,
                   best=best,best_rmse=best_rmse,stale=stale,history=history,python_rng=random.getstate(),numpy_rng=np.random.get_state(),
                   torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all() if amp else None,generator_rng=gen.get_state())
        torch.save(state,output/'resume.tmp'); os.replace(output/'resume.tmp',output/'resume.pt')
        dump(output/'history.json',history)
        report_progress(output,'checkpoint_saved',epoch=epoch+1,new_best=improved,
                        train_loss=history[-1]['train_loss'],dev_spearman=score,dev_rmse=rmse)
    metadata['elapsed_seconds_this_invocation']=time.perf_counter()-started
    metadata['best_epoch']=torch.load(output/'best.pt',map_location='cpu',weights_only=False)['epoch']
    dump(output/'run_metadata.json',metadata)
    model.load_state_dict(torch.load(output/'best.pt',map_location=device,weights_only=False)['model'])
    if not final:
        for split in ['dev']+(['test'] if evaluate_test else []):
            report_progress(output,'export',split=split,rows=len(subsets[split]))
            frame=prediction_frame(subsets[split],predict(model,datasets[split],config['batch_size'],device),variant,config['seed'],split,datasets[split].logs)
            export_bundle(output/split,frame,metadata,manifest,history)
    report_progress(output,'complete',best_epoch=metadata['best_epoch']+1,
                    elapsed_seconds=round(metadata['elapsed_seconds_this_invocation'],2),
                    metrics=str(output/'dev/metrics.json') if not final else None)
    return model,tokenizer,metadata


def predict_checkpoint(directory,records):
    import torch
    from transformers import AutoTokenizer,AutoConfig,AutoModel
    directory=Path(directory); meta=json.loads((directory/'run_metadata.json').read_text(encoding='utf-8'))
    c=meta['config']; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if records[0]['task']!=meta['task']: raise ValueError('Checkpoint task mismatch')
    tokenizer=AutoTokenizer.from_pretrained(directory/'tokenizer',use_fast=True,local_files_only=True)
    encoder=AutoModel.from_config(AutoConfig.from_pretrained(directory/'encoder_config',local_files_only=True))
    model=Regressor(c['backbone'],len(SCHEMAS[meta['task']]['labels']),meta['model'],layer=c['layer'],encoder=encoder).to(device)
    model.load_state_dict(torch.load(directory/'best.pt',map_location=device,weights_only=False)['model'])
    dataset=EncodedDataset(records,tokenizer,c,meta['model'])
    frame=prediction_frame(records,predict(model,dataset,c['batch_size'],device),meta['model'],c['seed'],'official_test',dataset.logs)
    return frame,meta


## 4. Đọc đúng dữ liệu và split đã khóa
Nhận cả Dataset đã giải nén và ZIP còn nguyên. Kiểm tra source/manifest SHA256, full ID coverage, target overlap và exact duplicates; không gọi make_splits.
Alignment được kiểm tra trên Kaggle; ambiguous/missing rows vẫn giữ, dùng expression fallback có log khi tokenize. Đọc coverage trước khi diễn giải context effect.


In [ ]:
import tempfile
records, manifest, DATA_PROVENANCE = load_frozen_task(DATA_ROOT, TASK, Path(tempfile.gettempdir())/'cosense_input_cache', split_seed=SPLIT_SEED)
print(json.dumps(DATA_PROVENANCE, indent=2))
print('Alignment:', dict(__import__('collections').Counter(r['alignment_status'] for r in records)))
RUN_ROOT.mkdir(parents=True, exist_ok=True)
dump(RUN_ROOT / 'data_provenance.json', DATA_PROVENANCE)


## 5. TRAIN — kiểm tra đúng model trước khi chạy
In setup/tokenize/epoch/batch/loss/dev metrics/checkpoint. Best checkpoint chọn bằng mean valid-output dev Spearman; undefined rho giữ NA và fallback dev RMSE theo policy.
LOAD REPORT có UNEXPECTED cls.* khi lấy BERT encoder là các pretraining heads không dùng; riêng cảnh báo đó không phải traceback lỗi.
Đừng chạy cell training chỉ để xem metrics cũ; dùng cell 6. Nếu bắt đầu run mới ở cùng folder, hãy giữ/save outputs cũ trước.


In [ ]:
if MODEL != 'B1' or NOTEBOOK_MODEL != 'B1' or CONFIG['seed'] != SEED or CONFIG['smoke'] != SMOKE:
    raise RuntimeError('Runtime settings không khớp. Chạy lại cell cấu hình B1.')
expected_run = OUTPUT_ROOT / ('smoke_runs' if SMOKE else 'runs') / TASK / f'B1_seed{SEED}'
if RUN_ROOT != expected_run: raise RuntimeError('Run path không khớp settings.')
print('TRAIN:', TASK, 'B1', 'SMOKE' if SMOKE else 'FULL', 'seed', SEED, flush=True)
model, tokenizer, metadata = train(records, manifest, CONFIG, 'B1', RUN_ROOT,
                                  resume=RESUME, evaluate_test=EVALUATE_LOCAL_TEST)
print('Training finished. Best epoch:', metadata['best_epoch']+1)
ARTIFACT_MANIFEST = publish_run_artifacts(RUN_ROOT, PUBLISH_ROOT, 'B1', TASK, SEED, smoke=SMOKE)


## 6. XEM METRICS — không train lại
Đọc dev metrics của best checkpoint. NN modifier/head riêng, PV Avg. Spearman cao hơn tốt hơn; RMSE/delta MAE thấp hơn tốt hơn. Variance correlation undefined → NA, không thêm noise.
Đây là dev để lựa chọn cấu hình. Local test chưa đánh giá nếu EVALUATE_LOCAL_TEST=False.


In [ ]:
import pandas as pd
metrics_path = RUN_ROOT / 'dev/metrics.json'
meta_path = RUN_ROOT / 'dev/run_metadata.json'
if not metrics_path.is_file() or not meta_path.is_file():
    raise FileNotFoundError(f'Chưa có dev metrics: {RUN_ROOT}. Đợi training/export xong hoặc chọn đúng run.')
saved_meta = json.loads(meta_path.read_text())
if saved_meta.get('model') != 'B1' or saved_meta.get('task') != TASK or saved_meta.get('seed') != SEED:
    raise ValueError('Metrics không đúng model/task/seed.')
if saved_meta.get('config',{}).get('smoke') != SMOKE: raise ValueError('Đang đọc nhầm smoke/full run.')
metrics = json.loads(metrics_path.read_text())
table = pd.DataFrame([dict(Model='B1', Task=TASK, Seed=SEED, Smoke=SMOKE, Split='dev', Output=o,
                          N=v['n_valid'], Spearman=v['spearman']['value'], RMSE=v['rmse'],
                          Variance_correlation=v['context_variance']['value'], Delta_MAE=v['delta_mae_macro'])
                      for o,v in metrics['outputs'].items()])
try:
    from IPython.display import display
    display(table)
except ImportError: print(table.to_string(index=False))
print('Metrics:', metrics_path)
print('Results bundle:', RUN_ROOT / 'dev/results_bundle.zip')
print('Keep all checkpoints/tokenizer:', RUN_ROOT)


## 7. KIỂM TRA / XUẤT OUTPUT CÓ SẴN — KHÔNG TRAIN LẠI
Cell này chỉ kiểm tra hoặc đóng gói files đã có. Nếu missing artifacts thì dừng và in lỗi, không tự train lại.
Files rõ tên nằm trực tiếp ở gốc /kaggle/working, không chỉ trong thư mục con. OUTPUTS_READY là bằng chứng files tồn tại trong session; vẫn phải lưu outputs vào version Kaggle.


In [ ]:
tag = f'{TASK.upper()}_B1_seed{SEED}_{"smoke" if SMOKE else "full"}'
ARTIFACT_MANIFEST = PUBLISH_ROOT / f'{tag}_ARTIFACTS.json'
if not ARTIFACT_MANIFEST.is_file():
    ARTIFACT_MANIFEST = publish_run_artifacts(RUN_ROOT, PUBLISH_ROOT, 'B1', TASK, SEED, smoke=SMOKE)
inventory = verify_published_artifacts(ARTIFACT_MANIFEST)
print('OUTPUTS_READY: verified', len(inventory['files']), 'files in', PUBLISH_ROOT)
print(pd.DataFrame(inventory['files']).to_string(index=False))
try:
    from IPython.display import FileLink, display
    for item in inventory['files']:
        display(FileLink(str(PUBLISH_ROOT/item['name'])))
except ImportError: pass


## 8. Save Version và giữ outputs

Save Version **kèm outputs**, tên gợi ý `NN-B1-full-seed42` (TASK=pv thì `PV-B1-full-seed42`).

- `NN_B1_seed42_full_results.zip` (hoặc PV): ngay ở gốc /kaggle/working, download/import web local.
- `NN_B1_seed42_full_model.zip`: best checkpoint + tokenizer/config để inference; không chứa optimizer resume state.
- `NN_B1_seed42_full_metrics.csv`, `..._history.csv`, `..._ARTIFACTS.json`: bảng metrics/history và danh sách artifact bytes ngay ở gốc Output.
- Cả run folder: `best.pt`, `resume.pt`, `tokenizer/`, `encoder_config/`, config/history/metadata/manifests/data_provenance để resume hoặc inference sau này.
- So B1/B2 phải cùng source_sha256, manifest_sha256, task, training seed và config/budget. Không chỉ so các run khác dữ liệu/split.

Resume: đặt RESUME tới checkpoint đã attach, giữ y hệt CONFIG/backbone string/planned epochs/task/model/manifest. Best.pt + tokenizer/encoder_config cần ở cạnh resume.pt. OUTPUT_ROOT phải writable; input read-only. Chỉ resume ở epoch boundary, không bảo đảm bitwise cross-device. Chỉ load checkpoint tin cậy của bạn.

Một seed là pilot; final experiments dùng ít nhất 42/43/44, giữ SPLIT_SEED=42. Không dùng official test để chọn model. Notebook không tự upload dữ liệu hoặc results.
